In [1]:
%pip install torch-pruning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 7.8 MB/s eta 0:00:00


In [2]:
import torch
import random
import copy
import os
import torch_pruning as tp

from torch.nn.utils import prune
from torchvision.models import resnet18
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from tqdm import tqdm

# Прунинг

В работе рассмотрены 4 метода:
1. магнитудный неструктурированный
2. магнитудный структурный
3. градиентный неструктурированный
4. градиентный структурный

Проведено сравнение со случайным удалением весов/каналов и
измерены метрики loss и accuracy для различных долей удаляемых весов/каналов

## Модель для измерения качества

In [3]:
seed = 1349
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_epochs = 5
generator = torch.manual_seed(seed)
random.seed(seed)



dataset = CIFAR10(
    root='data/',
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        )
    ])
)
dataset_test = CIFAR10(
    root='data/',
    train=False,
    download=False,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616),
        )
    ])
)
dataset_train, dataset_val = torch.utils.data.random_split(
    dataset,
    [0.8, 0.2],
    generator
)

dataloader_train = DataLoader(
    dataset_train,
    batch_size=64,
    shuffle=True,
    num_workers=1,
    generator=generator,
)
dataloader_val = DataLoader(
    dataset_val,
    batch_size=64,
    num_workers=1,
    shuffle=False
)
dataloader_test = DataLoader(
    dataset_test,
    batch_size=64,
    num_workers=1,
    shuffle=False
    )

samples_train = len(dataset_train)
samples_val = len(dataset_val)

model = resnet18(weights=None)
model.fc = torch.nn.Linear(512, 10)
model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=1)
model.maxpool = torch.nn.Identity()
model.to(device)

100%|██████████| 170M/170M [48:38<00:00, 58.4kB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1

In [4]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.002)


def train():
    for epoch in range(n_epochs):
        loss_train = 0
        loss_val = 0
        samples_correct = 0

        model.train()
        for i_batch_train, batch in enumerate(tqdm(dataloader_train)):
            images = batch[0].to(device)
            labels = batch[1].to(device)

            optimizer.zero_grad()

            output = model(images)

            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()

            loss_train += loss.item() * images.size(0)

        model.eval()
        with torch.no_grad():
            for i_batch_val, batch in enumerate(tqdm(dataloader_val)):
                images = batch[0].to(device)
                labels = batch[1].to(device)

                output = model(images)
                loss = criterion(output, labels)

                loss_val += loss.item() * images.size(0)
                samples_correct += (labels == output.argmax(dim=1)).sum().item()

        print(
            f' epoch: {epoch}\t'
            f'train loss: {loss_train / samples_train}\t'
            f'val loss: {loss_val / samples_val}\t'
            f'val accuracy: {samples_correct / samples_val} ({samples_correct}/{samples_val})'
        )

    torch.save(model.state_dict(), 'data/model.pth')


if __name__ == '__main__':
    if not os.path.exists('data/model.pth'):
        train()

100%|██████████| 157/157 [00:05<00:00, 29.56it/s]


 epoch: 0	train loss: 1.2294857056617736	val loss: 1.1025829383850099	val accuracy: 0.6221 (6221/10000)


100%|██████████| 157/157 [00:04<00:00, 32.76it/s]


 epoch: 1	train loss: 0.7599975516319275	val loss: 0.7359456311225892	val accuracy: 0.7426 (7426/10000)


100%|██████████| 157/157 [00:04<00:00, 33.53it/s]


 epoch: 2	train loss: 0.5673923938751221	val loss: 0.6238385698318482	val accuracy: 0.7841 (7841/10000)


100%|██████████| 157/157 [00:04<00:00, 32.93it/s]


 epoch: 3	train loss: 0.4236461318254471	val loss: 0.6017332517623901	val accuracy: 0.7992 (7992/10000)


100%|██████████| 157/157 [00:04<00:00, 32.75it/s]


 epoch: 4	train loss: 0.30180689581632614	val loss: 0.6680049005508423	val accuracy: 0.7876 (7876/10000)


## Магнитудный неструктурированный прунинг

In [5]:
class MagnitudePruner(prune.BasePruningMethod):
    PRUNING_TYPE = 'unstructured'

    def __init__(self, percentage, *params):
        super().__init__(*params)
        self.percentage = percentage

    def compute_mask(self, t, default_mask):
        weights = t.flatten().abs()
        weights, _ = torch.sort(weights)
        n_params = weights.shape[0]
        threshold = weights[int(n_params * self.percentage)]
        mask = t.abs() > threshold
        return mask * default_mask

In [6]:
model = torch.nn.Linear(5, 5)
print('before pruning:\n', list(model.named_parameters()))

MagnitudePruner.apply(model, name='weight', percentage=0.5)

print('after pruning:\n', model.weight)

before pruning:
 [('weight', Parameter containing:
tensor([[-0.0545,  0.1608, -0.1423, -0.0497,  0.2704],
        [-0.1947,  0.4398, -0.0302, -0.1817,  0.3252],
        [-0.0392, -0.0196, -0.1366, -0.0441, -0.0519],
        [ 0.0604, -0.0012,  0.0763, -0.2926,  0.0114],
        [ 0.4077,  0.0333,  0.1348, -0.0578,  0.2568]], requires_grad=True)), ('bias', Parameter containing:
tensor([-0.0242, -0.4389,  0.4449, -0.0676,  0.3047], requires_grad=True))]
after pruning:
 tensor([[-0.0000,  0.1608, -0.1423, -0.0000,  0.2704],
        [-0.1947,  0.4398, -0.0000, -0.1817,  0.3252],
        [-0.0000, -0.0000, -0.1366, -0.0000, -0.0000],
        [ 0.0000, -0.0000,  0.0000, -0.2926,  0.0000],
        [ 0.4077,  0.0000,  0.1348, -0.0000,  0.2568]], grad_fn=<MulBackward0>)


## Магнитудный структурный прунинг

In [7]:
class ExampleModel(torch.nn.Module):
    def __init__(self, in_size=3, hidden_size=5, out_size=3):
        super().__init__()
        self.hidden_size = hidden_size

        self.conv = torch.nn.Conv2d(in_size, hidden_size, kernel_size=3)
        self.pooling = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.relu = torch.nn.ReLU()
        self.fc1 = torch.nn.Linear(hidden_size, hidden_size)
        self.fc2 = torch.nn.Linear(hidden_size, out_size)

    def forward(self, x):
        x = self.pooling(self.relu(self.conv(x))).reshape(-1, self.hidden_size)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return x

In [8]:
class MagnitudeStructuredPruner:

    @staticmethod
    def apply(model, module, pruning_fn, percentage, example_inputs):
        dg = tp.DependencyGraph().build_dependency(
            model,
            example_inputs=example_inputs
        )

        t = module.weight.abs()
        t = t.reshape(t.size(0), -1)
        t = t.mean(dim=1)
        n_params = t.size(0)
        n_idxs = int(n_params * percentage)

        idxs = torch.topk(t, n_idxs, largest=False)[1].tolist()

        group = dg.get_pruning_group(module, pruning_fn, idxs=idxs)

        if dg.check_pruning_group(group):
            group.prune()

In [9]:
model = ExampleModel()
print('before pruning:', model)
MagnitudeStructuredPruner.apply(
    model,
    model.fc1,
    tp.prune_linear_out_channels,
    percentage=0.5,
    example_inputs=torch.randn(3, 3, 32, 32)
)
MagnitudeStructuredPruner.apply(
    model,
    model.conv,
    tp.prune_conv_out_channels,
    percentage=0.5,
    example_inputs=torch.randn(3, 3, 32, 32)
)

print('after pruning:', model)

before pruning: ExampleModel(
  (conv): Conv2d(3, 5, kernel_size=(3, 3), stride=(1, 1))
  (pooling): AdaptiveAvgPool2d(output_size=(1, 1))
  (relu): ReLU()
  (fc1): Linear(in_features=5, out_features=5, bias=True)
  (fc2): Linear(in_features=5, out_features=3, bias=True)
)
after pruning: ExampleModel(
  (conv): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1))
  (pooling): AdaptiveAvgPool2d(output_size=(1, 1))
  (relu): ReLU()
  (fc1): Linear(in_features=3, out_features=3, bias=True)
  (fc2): Linear(in_features=3, out_features=3, bias=True)
)


## Градиентный неструктурированный прунинг

In [10]:
n_classes = 5
n_images = 20

model = ExampleModel(hidden_size=4, out_size=n_classes)

example_inputs = torch.randn(n_images, 3, 32, 32)
example_labels = torch.randint(low=0, high=n_classes, size=(n_images,))
batch_size = 3

criterion_grad = torch.nn.CrossEntropyLoss(reduction='sum')
model.zero_grad()

for i in range(int(example_inputs.size(0) / batch_size) + 1):
    images = example_inputs[
        batch_size * i:
        min(batch_size * (i + 1), example_inputs.size(0))
    ]
    labels = example_labels[
        batch_size * i:
        min(batch_size * (i + 1), example_inputs.size(0))
    ]
    output = model(images)

    loss = criterion_grad(output, labels)
    loss.backward()

grad = model.fc1.weight.grad / n_images

print(grad)

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0052, -0.0073, -0.0077, -0.0088],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0073, -0.0091, -0.0112, -0.0126]])


In [11]:
class GradientPruner(prune.BasePruningMethod):
    PRUNING_TYPE = 'unstructured'

    def __init__(self, percentage, t_gradient, *params):
        super().__init__(*params)
        self.percentage = percentage
        self.t_gradient = t_gradient

    def compute_mask(self, t, default_mask):
        abs_grad_list = self.t_gradient.flatten().abs()
        abs_grad_list, _ = torch.sort(abs_grad_list)
        n_params = abs_grad_list.shape[0]
        threshold = abs_grad_list[int(n_params * self.percentage)]
        mask = self.t_gradient.abs() > threshold
        return mask * default_mask

In [12]:

GradientPruner.apply(model.fc1, name='weight', percentage=0.5, t_gradient=grad)

print('after pruning:\n', model.fc1.weight)

abs_grad_list = grad.flatten().abs()
abs_grad_list, _ = torch.sort(abs_grad_list)
n_params = abs_grad_list.shape[0]
threshold = abs_grad_list[int(n_params * 0.5)]
mask = grad.abs() > threshold

print('mask:\n', mask * 1)
print('gradient:\n', grad)
print('threshold:', threshold)

after pruning:
 tensor([[-0.0000, -0.0000, -0.0000, -0.0000],
        [ 0.0000,  0.3460, -0.4656,  0.4094],
        [-0.0000, -0.0000,  0.0000, -0.0000],
        [ 0.1519,  0.4535,  0.0950,  0.1261]], grad_fn=<MulBackward0>)
mask:
 tensor([[0, 0, 0, 0],
        [0, 1, 1, 1],
        [0, 0, 0, 0],
        [1, 1, 1, 1]])
gradient:
 tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0052, -0.0073, -0.0077, -0.0088],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0073, -0.0091, -0.0112, -0.0126]])
threshold: tensor(0.0052)


## Градиентный структурнный прунинг

In [13]:
class GradientStructuredPruner:

    @staticmethod
    def apply(model, module, pruning_fn, percentage, t_gradient, example_inputs):
        dg = tp.DependencyGraph().build_dependency(
            model,
            example_inputs=example_inputs
        )

        abs_grad = t_gradient.abs()
        abs_grad = abs_grad.reshape(abs_grad.shape[0], -1)
        abs_grad = abs_grad.mean(dim=1)

        n_params = abs_grad.size(0)
        n_idxs = int(n_params * percentage)

        idxs = torch.topk(abs_grad, n_idxs, largest=False)[1].tolist()

        group = dg.get_pruning_group(module, pruning_fn, idxs=idxs)

        if dg.check_pruning_group(group):
            group.prune()

In [14]:
n_classes = 5
n_images = 20

model = ExampleModel(hidden_size=4, out_size=n_classes)

example_inputs = torch.randn(n_images, 3, 32, 32)
example_labels = torch.randint(low=0, high=n_classes, size=(n_images,))
batch_size = 3

criterion_grad = torch.nn.CrossEntropyLoss(reduction='sum')
model.zero_grad()

for i in range(int(example_inputs.size(0) / 3) + 1):
    images = example_inputs[
        batch_size * i:
        min(batch_size * (i + 1), example_inputs.size(0))
    ]
    labels = example_labels[
        batch_size * i:
        min(batch_size * (i + 1), example_inputs.size(0))
    ]
    output = model(images)

    loss = criterion_grad(output, labels)
    loss.backward()


print('before pruning:', model)
GradientStructuredPruner.apply(
    model,
    model.fc1,
    tp.prune_linear_out_channels,
    percentage=0.5,
    t_gradient=model.fc1.weight.grad,
    example_inputs=torch.randn(3, 3, 32, 32)
)
GradientStructuredPruner.apply(
    model,
    model.conv,
    tp.prune_conv_out_channels,
    percentage=0.5,
    t_gradient=model.conv.weight.grad,
    example_inputs=torch.randn(3, 3, 32, 32)
)

print('after pruning:', model)

before pruning: ExampleModel(
  (conv): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1))
  (pooling): AdaptiveAvgPool2d(output_size=(1, 1))
  (relu): ReLU()
  (fc1): Linear(in_features=4, out_features=4, bias=True)
  (fc2): Linear(in_features=4, out_features=5, bias=True)
)
after pruning: ExampleModel(
  (conv): Conv2d(3, 2, kernel_size=(3, 3), stride=(1, 1))
  (pooling): AdaptiveAvgPool2d(output_size=(1, 1))
  (relu): ReLU()
  (fc1): Linear(in_features=2, out_features=2, bias=True)
  (fc2): Linear(in_features=2, out_features=5, bias=True)
)


## Пример использования с ResNet18

### Изначальные метрики на отложенной выборке

In [15]:
model = resnet18(weights=None)
model.fc = torch.nn.Linear(512, 10)
model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=1)
model.maxpool = torch.nn.Identity()

model.load_state_dict(
    torch.load(
        'data/model.pth',
        weights_only=True,
        map_location=device
    )
)
model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1

In [16]:
test_criterion = torch.nn.CrossEntropyLoss()


def test_model(model, dataloader, criterion):
    samples_total = 0
    samples_correct = 0
    loss_total = 0

    model.eval()
    with torch.no_grad():
        for i_batch, batch in enumerate(dataloader):
            images = batch[0].to(device)
            labels = batch[1].to(device)

            output = model(images)
            loss = criterion(output, labels)

            loss_total += loss.item() * images.size(0)
            samples_correct += (labels == output.argmax(dim=1)).sum().item()
            samples_total += images.size(0)

    return loss_total / samples_total, samples_correct / samples_total

In [17]:
loss, accuracy = test_model(model, dataloader_test, test_criterion)
print(f'loss: {loss}\t' f'accuracy: {accuracy}')

loss: 0.672844206905365	accuracy: 0.7853


### Магнитудный неструктурированый прунинг

In [18]:
for percentage in [0.05, 0.1, 0.15, 0.2, 0.25]:
    model_pruned_mu = copy.deepcopy(model)

    for module in model_pruned_mu.named_modules():
        module_name = module[0]
        if 'conv' not in module_name and 'fc' not in module_name:
            continue

        MagnitudePruner.apply(module[1], name='weight', percentage=percentage)

    print(f'\n{percentage = }')
    loss, accuracy = test_model(model_pruned_mu, dataloader_test, test_criterion)
    print(f'loss: {loss}\t' f'accuracy: {accuracy}')


percentage = 0.05
loss: 0.6737067174911499	accuracy: 0.7856

percentage = 0.1
loss: 0.6768764179229736	accuracy: 0.7832

percentage = 0.15
loss: 0.6922137025833129	accuracy: 0.7773

percentage = 0.2
loss: 0.7204931794166565	accuracy: 0.7693

percentage = 0.25
loss: 0.7531487995147705	accuracy: 0.7594


### Магнитудный структурный прунинг

Будем подвергать удалению только выходные каналы сверточных слоев. Это приведет к удалению входных каналов следующего слоя, поэтому отдельно удалять входные каналы не нужно.

По этой же причине изменять полносвязный слой не нужно. Число выходных каналов должно соответствовать числу классов.

In [19]:
example_inputs = torch.randn(3, 3, 32, 32).to(device)

for percentage in [0.02, 0.05, 0.08, 0.11, 0.14]:
    model_pruned_ms = copy.deepcopy(model)

    for module in model_pruned_ms.named_modules():
        module_name = module[0]
        if 'conv' not in module_name:
            continue

        MagnitudeStructuredPruner.apply(
            model_pruned_ms,
            module[1],
            tp.prune_conv_out_channels,
            percentage=percentage,
            example_inputs=example_inputs
        )

    print(f'\n{percentage = }')
    loss, accuracy = test_model(model_pruned_ms, dataloader_test, test_criterion)
    print(f'loss: {loss}\t' f'accuracy: {accuracy}')



percentage = 0.02
loss: 0.9707183433532715	accuracy: 0.7088

percentage = 0.05
loss: 1.8245739753723145	accuracy: 0.489

percentage = 0.08
loss: 4.18018931274414	accuracy: 0.3021

percentage = 0.11
loss: 5.210877545166015	accuracy: 0.1152

percentage = 0.14
loss: 5.196238986206055	accuracy: 0.1018


### Градиентный неструктурированный прунинг

In [21]:
criterion_grad = torch.nn.CrossEntropyLoss(reduction='sum')
model.zero_grad()

for i_batch_val, batch in enumerate(tqdm(dataloader_val)):
    images = batch[0].to(device)
    labels = batch[1].to(device)

    output = model(images)
    loss = criterion_grad(output, labels)
    loss.backward()

100%|██████████| 157/157 [00:07<00:00, 20.11it/s]


In [22]:
for percentage in [0.05, 0.1, 0.15, 0.2, 0.25]:
    model_pruned_gu = copy.deepcopy(model)

    for module in model_pruned_gu.named_modules():
        module_name = module[0]
        if 'conv' not in module_name and 'fc' not in module_name:
            continue

        GradientPruner.apply(
            module[1],
            name='weight',
            percentage=percentage,
            t_gradient=model.get_submodule(module_name).weight.grad
        )
    print(f'\n{percentage = }')
    loss, accuracy = test_model(model_pruned_gu, dataloader_test, test_criterion)
    print(f'loss: {loss}\t' f'accuracy: {accuracy}')


percentage = 0.05
loss: 0.7933384534835816	accuracy: 0.7393

percentage = 0.1
loss: 1.116563931274414	accuracy: 0.6238

percentage = 0.15
loss: 1.8553527751922607	accuracy: 0.4217

percentage = 0.2
loss: 2.758431953430176	accuracy: 0.1929

percentage = 0.25
loss: 3.0881790462493894	accuracy: 0.1681


### Градиентный структурный прунинг

In [23]:
for percentage in [0.02, 0.05, 0.08, 0.11, 0.14]:
    model_pruned_gs = copy.deepcopy(model)

    for module in model_pruned_gs.named_modules():
        module_name = module[0]
        if 'conv' not in module_name:
            continue

        GradientStructuredPruner.apply(
            model_pruned_gs,
            module[1],
            tp.prune_conv_out_channels,
            percentage=percentage,
            example_inputs=example_inputs,
            t_gradient=model.get_submodule(module_name).weight.grad
        )

    print(f'\n{percentage = }')
    loss, accuracy = test_model(model_pruned_gs, dataloader_test, test_criterion)
    print(f'loss: {loss}\t' f'accuracy: {accuracy}')


percentage = 0.02
loss: 0.9819922958374023	accuracy: 0.6997

percentage = 0.05
loss: 2.333008203125	accuracy: 0.4186

percentage = 0.08
loss: 3.8991425994873046	accuracy: 0.1772

percentage = 0.11
loss: 2.9361206184387205	accuracy: 0.2082

percentage = 0.14
loss: 4.1894392379760745	accuracy: 0.1574


### Убедимся, что реализованные методы лучше, чем случайное удаление весов

In [24]:
for percentage in [0.05, 0.1, 0.15, 0.2, 0.25]:
    n_tries = 3
    loss = 0
    accuracy = 0

    for _ in range(n_tries):
        model_pruned_ru = copy.deepcopy(model)

        for module in model_pruned_ru.named_modules():
            module_name = module[0]
            if 'conv' not in module_name and 'fc' not in module_name:
                continue

            prune.random_unstructured(module[1], name='weight', amount=percentage)

        result = test_model(model_pruned_ru, dataloader_test, test_criterion)
        loss += result[0]
        accuracy += result[1]

    print(f'\n{percentage = }')
    print(f'loss: {loss / n_tries}\t' f'accuracy: {accuracy / n_tries}')


percentage = 0.05
loss: 1.2032229952494304	accuracy: 0.6256

percentage = 0.1
loss: 2.8086669429779056	accuracy: 0.33813333333333334

percentage = 0.15
loss: 3.3626777992248535	accuracy: 0.24683333333333332

percentage = 0.2
loss: 3.9837502801259355	accuracy: 0.15059999999999998

percentage = 0.25
loss: 4.057931506856282	accuracy: 0.10373333333333333


In [25]:
class RandomStructuredPruner:

    @staticmethod
    def apply(model, module, pruning_fn, percentage, example_inputs):
        dg = tp.DependencyGraph().build_dependency(
            model,
            example_inputs=example_inputs
        )

        n = module.weight.size(0)
        idxs = torch.randperm(n)[:int(n * percentage)].tolist()

        group = dg.get_pruning_group(module, pruning_fn, idxs=idxs)

        if dg.check_pruning_group(group):
            group.prune()

In [26]:
for percentage in [0.02, 0.05, 0.08, 0.11, 0.14]:
    n_tries = 3

    loss = 0
    accuracy = 0

    for _ in range(n_tries):
        model_pruned_rs = copy.deepcopy(model)
        for module in model_pruned_rs.named_modules():
            module_name = module[0]
            if 'conv' not in module_name:
                continue

            RandomStructuredPruner.apply(
                model_pruned_rs,
                module[1],
                tp.prune_conv_out_channels,
                percentage=percentage,
                example_inputs=example_inputs,
            )
        result = test_model(model_pruned_rs, dataloader_test, test_criterion)
        loss += result[0]
        accuracy += result[1]

    print(f'\n{percentage = }')
    print(f'loss: {loss / n_tries}\t' f'accuracy: {accuracy / n_tries}')


percentage = 0.02
loss: 0.8852053560574848	accuracy: 0.7196666666666666

percentage = 0.05
loss: 1.7917712626775106	accuracy: 0.4888666666666667

percentage = 0.08
loss: 3.0172249209086104	accuracy: 0.29843333333333333

percentage = 0.11
loss: 4.494452554957072	accuracy: 0.13393333333333332

percentage = 0.14
loss: 4.2195571240743	accuracy: 0.11606666666666665
